# CatBoost 재튜닝 (Optuna 30trial + 3-Fold 탐색 → 5-Fold 최종 학습)

torch를 전혀 사용하지 않는 독립 노트북입니다. LightGBM/TabTransformer와
스레드를 공유하지 않으므로 충돌 걱정 없이 정상 속도로 동작합니다.

**사전 준비**: `uv add catboost optuna` (둘 다 아직 설치 안 되어 있다면)

**예상 시간**: 탐색 단계(30trial × 3-fold) + 최종 5-fold 학습 합쳐서 대략 30~60분
(CatBoost는 fold당 보통 30~70초 정도 걸립니다)

## 1. 라이브러리 및 설정

In [1]:
import os
import json
import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_TRIALS = 30
SEARCH_SPLITS = 3
FINAL_SPLITS = 5
RANDOM_STATE = 1004
CACHE_DIR = "blend_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

## 2. 데이터 전처리 (main.py와 동일 + CatBoost 전용 처리)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))

# CatBoost는 LabelEncoder로 정수화하지 않고 범주형을 문자열 그대로 cat_features로
# 넘기는 게 보통 더 성능이 좋습니다 (자체 타겟 기반 인코딩을 내부적으로 사용).
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    df[col] = df[col].astype(str)

y = df[TARGET_COL].values.astype(int)
X = df.drop(columns=[TARGET_COL])

print(f"전처리 완료: {X.shape}, 범주형 피처 {len(cat_cols)}개")

전처리 완료: (256351, 67), 범주형 피처 20개


## 3. Optuna 탐색 (3-Fold, 30 trial)

In [3]:
def objective(trial):
    params = {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
    }

    skf = StratifiedKFold(n_splits=SEARCH_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = CatBoostClassifier(
            iterations=1500,
            **params,
            cat_features=cat_cols,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )
        model.fit(
            X.iloc[tr_idx], y[tr_idx],
            eval_set=(X.iloc[val_idx], y[val_idx]),
            early_stopping_rounds=50,
        )
        preds = model.predict_proba(X.iloc[val_idx])[:, 1]
        fold_scores.append(roc_auc_score(y[val_idx], preds))

    return float(np.mean(fold_scores))


def print_progress(study, trial):
    print(f"Trial {trial.number + 1}/{N_TRIALS}  AUC={trial.value:.5f}  (현재 최고: {study.best_value:.5f})")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, callbacks=[print_progress])

print("\n최적 파라미터:")
print(study.best_params)
print(f"탐색 단계 최고 AUC (3-fold): {study.best_value:.5f}")

Trial 1/30  AUC=0.73926  (현재 최고: 0.73926)
Trial 2/30  AUC=0.73954  (현재 최고: 0.73954)
Trial 3/30  AUC=0.73978  (현재 최고: 0.73978)
Trial 4/30  AUC=0.73867  (현재 최고: 0.73978)
Trial 5/30  AUC=0.73942  (현재 최고: 0.73978)
Trial 6/30  AUC=0.73978  (현재 최고: 0.73978)
Trial 7/30  AUC=0.73901  (현재 최고: 0.73978)
Trial 8/30  AUC=0.73828  (현재 최고: 0.73978)
Trial 9/30  AUC=0.73925  (현재 최고: 0.73978)
Trial 10/30  AUC=0.73960  (현재 최고: 0.73978)
Trial 11/30  AUC=0.73920  (현재 최고: 0.73978)
Trial 12/30  AUC=0.73969  (현재 최고: 0.73978)
Trial 13/30  AUC=0.73990  (현재 최고: 0.73990)
Trial 14/30  AUC=0.73980  (현재 최고: 0.73990)
Trial 15/30  AUC=0.73973  (현재 최고: 0.73990)
Trial 16/30  AUC=0.73965  (현재 최고: 0.73990)
Trial 17/30  AUC=0.73951  (현재 최고: 0.73990)
Trial 18/30  AUC=0.73971  (현재 최고: 0.73990)
Trial 19/30  AUC=0.73968  (현재 최고: 0.73990)
Trial 20/30  AUC=0.73977  (현재 최고: 0.73990)
Trial 21/30  AUC=0.73971  (현재 최고: 0.73990)
Trial 22/30  AUC=0.73954  (현재 최고: 0.73990)
Trial 23/30  AUC=0.73954  (현재 최고: 0.73990)
Trial 24/30  AUC=0.7

## 4. 최적 파라미터로 5-Fold 최종 학습

In [4]:
best_params = study.best_params

skf = StratifiedKFold(n_splits=FINAL_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_catboost = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    model = CatBoostClassifier(
        iterations=3000,
        **best_params,
        cat_features=cat_cols,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )
    model.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=(X.iloc[val_idx], y[val_idx]),
        early_stopping_rounds=100,
    )
    oof_catboost[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y[val_idx], oof_catboost[val_idx])
    print(f"Fold {fold}/{FINAL_SPLITS}  CatBoost AUC: {fold_auc:.5f}  (best_iteration: {model.get_best_iteration()})")

score_catboost = roc_auc_score(y, oof_catboost)
print(f"\nCatBoost 5-Fold 전체 OOF AUC: {score_catboost:.5f}")

Fold 1/5  CatBoost AUC: 0.74351  (best_iteration: 1749)
Fold 2/5  CatBoost AUC: 0.74048  (best_iteration: 1685)
Fold 3/5  CatBoost AUC: 0.73751  (best_iteration: 1436)
Fold 4/5  CatBoost AUC: 0.73811  (best_iteration: 918)
Fold 5/5  CatBoost AUC: 0.73912  (best_iteration: 1011)

CatBoost 5-Fold 전체 OOF AUC: 0.73971


## 5. 결과 저장

In [5]:
with open("catboost_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, ensure_ascii=False, indent=2)

np.save(f"{CACHE_DIR}/oof_catboost.npy", oof_catboost)
np.save(f"{CACHE_DIR}/oof_y.npy", y.astype(np.float32))

print("저장 완료: catboost_best_params.json, blend_cache/oof_catboost.npy")
print(f"\n비교: 현재 main.py LightGBM 튜닝 점수 = 0.7400")
print(f"CatBoost 재튜닝 점수 = {score_catboost:.5f}")
print(f"차이: {score_catboost - 0.7400:+.5f}")

저장 완료: catboost_best_params.json, blend_cache/oof_catboost.npy

비교: 현재 main.py LightGBM 튜닝 점수 = 0.7400
CatBoost 재튜닝 점수 = 0.73971
차이: -0.00029


## 6. (보너스) LightGBM과의 블렌딩 가능성 빠르게 확인

In [ ]:
try:
    oof_lgbm = np.load(f"{CACHE_DIR}/oof_lgbm.npy")
    score_lgbm = roc_auc_score(y, oof_lgbm)
    corr = np.corrcoef(oof_catboost, oof_lgbm)[0, 1]

    print(f"LightGBM 단독: {score_lgbm:.5f}")
    print(f"CatBoost 단독: {score_catboost:.5f}")
    print(f"OOF 상관계수: {corr:.4f}")
    print()

    baseline = max(score_lgbm, score_catboost)
    best_w, best_score = (1.0 if score_lgbm >= score_catboost else 0.0), baseline
    for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        blend = w * oof_lgbm + (1 - w) * oof_catboost
        blend_score = roc_auc_score(y, blend)
        marker = " <-- 현재 최고" if blend_score > best_score else ""
        if blend_score > best_score:
            best_w, best_score = w, blend_score
        print(f"LightGBM {w:.1f} + CatBoost {1-w:.1f}: {blend_score:.5f}{marker}")
 
    print(f"\n최적 가중치: LightGBM {best_w:.1f} + CatBoost {1-best_w:.1f} (점수: {best_score:.5f})")
    print(f"단독 최고 대비 개선: {best_score - baseline:+.5f}")
except FileNotFoundError:
    print("blend_cache/oof_lgbm.npy가 없습니다. 1차_lgbm_oof_only.ipynb를 먼저 실행하면 비교할 수 있습니다.")

LightGBM 단독: 0.73925
CatBoost 단독: 0.73971
OOF 상관계수: 0.9912

LightGBM 0.0 + CatBoost 1.0: 0.73971
LightGBM 0.1 + CatBoost 0.9: 0.73984 <-- 현재 최고
LightGBM 0.2 + CatBoost 0.8: 0.73994 <-- 현재 최고
LightGBM 0.3 + CatBoost 0.7: 0.74000 <-- 현재 최고
LightGBM 0.4 + CatBoost 0.6: 0.74001 <-- 현재 최고
LightGBM 0.5 + CatBoost 0.5: 0.73999
LightGBM 0.6 + CatBoost 0.4: 0.73993
LightGBM 0.7 + CatBoost 0.3: 0.73982
LightGBM 0.8 + CatBoost 0.2: 0.73967
LightGBM 0.9 + CatBoost 0.1: 0.73948
LightGBM 1.0 + CatBoost 0.0: 0.73925

최적 가중치: LightGBM 0.4 + CatBoost 0.6 (점수: 0.74001)
단독 최고 대비 개선: +0.00031
